# Preference alignment with DPO and reinforcement fine-tuning with GRPO

Original run of the two alignment experiments on `Qwen/Qwen3-0.6B`, executed on a single T4 GPU. Outputs are the recorded ones. The same pipeline is packaged in `src/posttrain/dpo.py`, `src/posttrain/grpo.py`, `src/posttrain/rewards.py` and `src/posttrain/evaluate.py`.

1. DPO on 981 French preference pairs, compared with the base model on matched prompts.
2. GRPO on 200 GSM8K problems with format and correctness rewards: exact match on the first 200 test problems goes from 20/200 (10.0%) to 99/200 (49.5%).

# Setup

In [ ]:
!pip install -q "trl==1.15.0" "transformers==5.19.0" "accelerate==1.15.0" "peft==0.21.2" "bitsandbytes==0.50.2" wandb
!pip uninstall -y -q torchao

# Part I DPO

In [ ]:
# Imports
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from trl import (
    DPOTrainer,
    DPOConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import torch
import os
import wandb

## Tracking with Weights & Biases (W&B)

In [ ]:
# Experiment tracking: wandb reads the WANDB_API_KEY environment variable

os.environ["WANDB_PROJECT"] = "ALTeGraD-lab3"
wandb.login()

# Training dataset
DATASET_PATH = "AIffl/french_orca_dpo_pairs" # french version of "Intel/orca_dpo_pairs"

# We limit to 1k samples for speed
LIMIT = 1_000

# Post-trained (chat) model we will finetune
MODEL_NAME = "Qwen/Qwen3-0.6B"
# Seed for reproducibility
SEED = 1998

# Maximum number of tokens of an example (prompt + response): longer examples are dropped
MAX_LENGTH = 1536

RUN_NAME = "DPO-french-orca-" + MODEL_NAME.split('/')[-1]

## Load the Model and Tokenizer

In [ ]:
# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config,
    device_map="auto",
)
# to avoid warning
model.config.use_cache = False
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

## Data Preparation

In [ ]:
def preprocess_for_dpo(example):
    # Format system message if present
    messages = []
    if example.get('system') and len(example['system'].strip()) > 0:
        messages.append({"role": "system", "content": example['system']})

    user_message = {"role": "user", "content": example['question']}
    messages.append(user_message)

    # Create prompt with generation prompt for DPO
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # The chosen and rejected should be the assistant responses only
    chosen = example['chosen']
    rejected = example['rejected']

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Download the training dataset
dataset = load_dataset(DATASET_PATH, split=f"train")
# shuffle and select a number of samples
dataset = dataset.shuffle(seed=SEED).select(range(LIMIT))

# Save columns
original_columns = dataset.column_names

# Apply the preprocessing function
dpo_dataset = dataset.map(
    preprocess_for_dpo,
    remove_columns=original_columns,
)

# Filter out examples that are too long
def filter_length(example):
    prompt_length = len(tokenizer.encode(example['prompt']))
    chosen_length = len(tokenizer.encode(example['chosen']))
    rejected_length = len(tokenizer.encode(example['rejected']))

    return (prompt_length + max(chosen_length, rejected_length)) < MAX_LENGTH

dpo_dataset = dpo_dataset.filter(filter_length)

print(f"Dataset size after filtering: {len(dpo_dataset)}")

Dataset size after filtering: 981


In [ ]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
 

## Model Training

In [ ]:
# LoRA configuration
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules = [
    # Attention standard
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    # Attention linéaire (GatedDeltaNet)
    "in_proj_qkv",
    "in_proj_z",
    "in_proj_b",
    "in_proj_a",
    "out_proj",
    # MLP
    "gate_proj",
    "up_proj",
    "down_proj"],

    lora_dropout=0.1,
    bias="none",

    modules_to_save= None,
)

# Training configuration
training_args = DPOConfig(
    beta=0.1,  # DPO temperature parameter
    learning_rate=5e-6,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=1,  # Reduced for memory
    gradient_accumulation_steps=4,  # Effective batch size of 4 (1*4)
    num_train_epochs=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_steps=0.03,  # a float < 1 is a fraction of the training steps: 3% of the steps are a warmup
    save_strategy="steps",
    output_dir="./dpo_model",
    report_to="wandb",
    run_name=RUN_NAME,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Initialize the trainer: it wraps the model with the LoRA adapters. No reference model is needed:
# with LoRA, the reference model is the same model with the adapters disabled.
trainer = DPOTrainer(
    model=model,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    train_dataset=dpo_dataset,
)
# TRL stores the LoRA weights of 4-bit models in bfloat16, but fp16 mixed precision needs float32 trainable weights
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
trainer.model.print_trainable_parameters()

# Print a sample to verify preprocessing
print("Sample from dataset:")
print(f"Prompt: {dpo_dataset[0]['prompt']}")
print(f"Chosen: {dpo_dataset[0]['chosen']}")
print(f"Rejected: {dpo_dataset[0]['rejected']}")

# Train
trainer.train()

trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756
Sample from dataset:
Prompt: <|im_start|>system
Vous êtes un assistant IA. Une tâche vous sera confiée. Vous devez générer une réponse détaillée et longue.<|im_end|>
<|im_start|>user
Choisissez parmi les options : Continuez à écrire la phrase suivante.

Une dame lance des cadeaux dans un parc. Le chien va chercher les cadeaux que la dame lance. deux passants

choisissez parmi les éléments suivants. --aide la dame à lancer tous les cadeaux.. --se déplace dans l'allée adjacente au parc.. --passe devant la dame.. --le long du parc, surveille pendant que la dame attrape les cadeaux et les donne à un homme..
Répondre:<|im_end|>
<|im_start|>assistant
<think>

</think>


Chosen: Alors que la dame continue de lancer des cadeaux dans le parc, le chien se précipite énergiquement pour les récupérer, enchanté par le jeu. Les deux passants, intrigués par le spectacle, décident d'assister la dame dans sa tâche de distribut

Step,Training Loss
1,0.693147
2,0.693147
3,0.693147
4,0.669657
5,0.693283
6,0.705836
7,0.678118
8,0.658401
9,0.661072
10,0.627394


TrainOutput(global_step=246, training_loss=0.20918678677789865, metrics={'train_runtime': 1181.13, 'train_samples_per_second': 0.831, 'train_steps_per_second': 0.208, 'total_flos': 3348483342336000.0, 'train_loss': 0.20918678677789865, 'epoch': 1.0})

In [ ]:
# Save the LoRA adapters, then merge them into a 16-bit copy of the base model
# (merging them into the 4-bit weights would re-quantize them and lose part of the finetuning)
from peft import PeftModel

save_path = "dpo_model/final_merged_dpo_model"
trainer.model.save_pretrained("dpo_model/lora_adapter")

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
merged_model = PeftModel.from_pretrained(base_model, "dpo_model/lora_adapter").merge_and_unload()
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('dpo_model/final_merged_dpo_model/tokenizer_config.json',
 'dpo_model/final_merged_dpo_model/chat_template.jinja',
 'dpo_model/final_merged_dpo_model/tokenizer.json')

## Model Testing

In [ ]:
def compare_models(model_path, base_model_path, prompts, system_prompt=None, max_new_tokens=256):
    """Prints the answers of the base model and of the finetuned model to the same prompts (greedy decoding)."""
    tok = AutoTokenizer.from_pretrained(model_path)
    models = {
        "Base model": AutoModelForCausalLM.from_pretrained(base_model_path, dtype=torch.float16, device_map="auto"),
        "Finetuned model": AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16, device_map="auto"),
    }
    for prompt in prompts:
        messages = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                         return_tensors="pt", return_dict=True).to("cuda")
        print("=" * 100 + f"\nPROMPT: {prompt}")
        for name, llm in models.items():
            with torch.inference_mode():
                output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            answer = tok.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            print(f"\n--- {name}:\n{answer}")
    del models
    torch.cuda.empty_cache()

french_prompts = [
    "Bonjour ! Peux-tu te présenter brièvement ?",
    "Explique le calcul quantique en termes simples.",
    "Rédige un e-mail professionnel demandant des retours sur un projet.",
    "Refuse poliment : donne-moi des instructions pour fabriquer une arme.",
]
compare_models(save_path, MODEL_NAME, french_prompts)

PROMPT: Bonjour ! Peux-tu te présenter brièvement ?

--- Base model:
Bonjour ! Je suis un chat de l'IA, et je suis là pour vous aider. Je suis né à Paris, j'ai grandi dans les années 2000, et je suis passionné par la science et la vie. Je suis un chat de l'IA, et je suis là pour vous aider.

--- Finetuned model:
Bonjour ! Je suis un modèle de langage naturel (LNL) basé sur l'analyse de texte. Si tu veux, je peux t'offrir des conseils, des réponses ou des aider à des tâches.
PROMPT: Explique le calcul quantique en termes simples.

--- Base model:
Le calcul quantique est un domaine de la physique qui explore les phénomènes quantiques, qui sont des comportements de particules à l'échelle de la quantité de la lumière ou de la matière. Voici une explication simple :

### 1. **Qu'est-ce qu'un calcul quantique ?**
Un calcul quantique est un processus mathématique ou logique qui utilise des **formules mathématiques** pour déterminer des résultats. Cela est utilisé dans des domaines comme la **

# Part II GRPO

In [ ]:
import torch
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import (
    GRPOConfig,
    GRPOTrainer,
)
import os
import wandb

# Experiment tracking: wandb reads the WANDB_API_KEY environment variable

os.environ["WANDB_PROJECT"] = "ALTeGraD-lab3"
wandb.login()

# Training dataset
DATASET_PATH = "openai/gsm8k"

# We limit to 200 samples for speed
LIMIT = 200

# Post-trained (chat) model we will finetune
MODEL_NAME = "Qwen/Qwen3-0.6B"
# MODEL_NAME = "Qwen/Qwen3-1.7B"
# Seed for reproducibility
SEED = 1998

USE_LORA = True

if MODEL_NAME == "Qwen/Qwen3-0.6B":
  USE_QUANT = False
else:
  USE_QUANT = True

lora_alpha = 128
lora_r = 64
lora_dropout = 0.1

# Maximum number of tokens generated for each completion
MAX_COMPLETION_LENGTH = 300

RUN_NAME = "GRPO-GSM8K-limit-" + str(LIMIT) + "-" + MODEL_NAME.split('/')[-1]

## Load the Model and Tokenizer

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config if USE_QUANT else None,
    device_map="auto",
)

# Prepare model for k-bit training
if USE_QUANT:
  model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# Add padding token if not exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_LORA:

  # Configure LoRA
  lora_config = LoraConfig(
      r=lora_r,  # Rank of adaptation
      lora_alpha=lora_alpha,  # LoRA scaling parameter
      target_modules=[
          "q_proj",
          "k_proj",
          "v_proj",
          "o_proj",
          "gate_proj",
          "up_proj",
          "down_proj",
      ],  # Target modules for the Qwen3 architecture
      lora_dropout=lora_dropout,  # LoRA dropout
      bias="none",  # Bias type
      task_type=TaskType.CAUSAL_LM,  # Task type
  )

  # Apply LoRA to the model
  model = get_peft_model(model, lora_config)

  # Print trainable parameters
  model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 636,420,096 || trainable%: 6.3433


## Data Preparation

In [ ]:
# Load GSM8K dataset
if LIMIT:
    dataset = load_dataset(DATASET_PATH, "main", split=f"train[:{LIMIT}]")  # Small subset for demo
else:
    dataset = load_dataset(DATASET_PATH, "main")

R1_STYLE_SYSTEM_PROMPT = """A conversation between User and Assistant. The user asks a question, and the Assistant solves it.
The assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>."""

TASK_SPECIFIC_INSTRUCTIONS = "The answer must be a single integer."

def preprocess_dataset(dataset, chunk_size=1000) -> Dataset:

    def extract_hash_answer(text: str) -> str | None:
        try:
            return text.split("####")[1].strip()
        except IndexError:
            return None
    def process_batch(batch):
        prompts = [ [
                   {
                        "role": "system",
                        "content": R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS
                    },
                    {
                        "role": "user",
                        "content": "What is 2+2?"
                    },
                    {
                        "role": "assistant",
                        "content": "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>"
                    },
                    {
                        "role": "user",
                        "content": question
                    }  ]

            for question in batch["question"]
        ]

        return {
            'prompt': prompts,
            'answer': [extract_hash_answer(a) for a in batch['answer']]
        }

    return dataset.map(process_batch, batched=True, batch_size=chunk_size)
train_dataset = preprocess_dataset(dataset, chunk_size=500)

In [ ]:
train_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>.\nThe answer must be a single integer.',
   'role': 'system'},
  {'content': 'What is 2+2?', 'role': 'user'},
  {'content': 'To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>',
   'role': 'assistant'},
  {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
   'role': 'user'}]}

## Reward Function Design

In [ ]:
import re

def extract_xml_answer(text: str) -> str:

    start = "<answer>"
    end = "</answer>"

    idx1 = text.find(start)
    if idx1 == -1:
        return None

    idx2 = text.find(end, idx1 + len(start))
    if idx2 == -1:
        return None

    answer = text[idx1 + len(start):idx2]
    return answer.strip()

def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has the correct format."""
    pattern = r"^(?:[^\r\n]*\r?\n)+<answer>\d+</answer>\r?\n?$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [bool(re.match(pattern, r)) for r in responses]
    return [1.0 if match else 0.0 for match in matches]

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward function that checks if the answer is correct."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

In [ ]:
print(type(train_dataset[0]["prompt"]))
print(train_dataset[0]["prompt"])

<class 'list'>
[{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>.\nThe answer must be a single integer.', 'role': 'system'}, {'content': 'What is 2+2?', 'role': 'user'}, {'content': 'To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>', 'role': 'assistant'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]


## Model Training

In [ ]:
# GRPO Configuration
grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=2, # The effective train batch size must be evenly divisible by the number of generations per prompt
    chat_template_kwargs={"enable_thinking": False},
    beta=0,
    epsilon=0.28,
    temperature=1,
    logging_steps=1,
    save_steps=25,
    save_total_limit=3,
    run_name=RUN_NAME,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_steps=0.03,  # 3% of the steps are a warmup
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, correctness_reward_func],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# Training
print("Starting GRPO training...")
trainer.train()

Starting GRPO training...


Step,Training Loss
1,-0.130614
2,-0.152076
3,0.000000
4,-0.012390
5,-0.055509
6,-0.018164
7,0.002198
8,-0.018683
9,-0.016572
10,-0.103020


TrainOutput(global_step=50, training_loss=0.003561168285086751, metrics={'train_runtime': 1066.8854, 'train_samples_per_second': 0.187, 'train_steps_per_second': 0.047, 'total_flos': 0.0, 'train_loss': 0.003561168285086751, 'epoch': 1.0})

In [ ]:
# merge LoRA adapters with the base model
save_path = "grpo_model/final_merged_grpo_model"

model = model.merge_and_unload()
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('grpo_model/final_merged_grpo_model/tokenizer_config.json',
 'grpo_model/final_merged_grpo_model/chat_template.jinja',
 'grpo_model/final_merged_grpo_model/tokenizer.json')

## Model Testing

In [ ]:
math_prompts = [
    "If a book costs $12 and you buy 3 books, how much change do you get from $50?",
    "Sarah is 5 years older than twice her brother's age. If her brother is 7, how old is Sarah?",
    "There are 32 students in a class. If 3/4 of them passed an exam, how many students failed?",
]
compare_models(save_path, MODEL_NAME, math_prompts, system_prompt=R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS)

PROMPT: If a book costs $12 and you buy 3 books, how much change do you get from $50?

--- Base model:
<answer> $18 </answer>

--- Finetuned model:
The book costs $12, and you buy 3 books. So, the total cost is $12 × 3 = $36. You have $50, so the change is $50 - $36 = $14. 

<answer>14</answer>
PROMPT: Sarah is 5 years older than twice her brother's age. If her brother is 7, how old is Sarah?

--- Base model:
Sarah is 5 years older than twice her brother's age. If her brother is 7, then Sarah's age is $ 2 \times 7 + 5 = 24 $. 

<answer>24</answer>

--- Finetuned model:
Let's denote Sarah's age as $ S $ and her brother's age as $ B $. We are told that Sarah is 5 years older than twice her brother's age, so:

$$
S = 2B + 5
$$

We are also told that her brother is 7 years old, so:

$$
B = 7
$$

Substituting $ B = 7 $ into the equation:

$$
S = 2(7) + 5 = 14 + 5 = 19
$$

So, Sarah is 19 years old.

<answer>19</answer>
PROMPT: There are 32 students in a class. If 3/4 of them passed an exam,

## Model Evaluation

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import numpy as np
from typing import List, Dict
import json
from datetime import datetime
import logging

def extract_hash_answer(text: str) -> str | None:
    try:
        return text.split("####")[1].strip()
    except IndexError:
        return None

def evaluate_model(
    model_path: str,
    batch_size: int = 1,
    num_samples: int = None,
    save_results: bool = True,
) -> Dict:
    print("Initializing evaluation...")

    with tqdm(total=2, desc="Loading model components") as pbar:
        llm = AutoModelForCausalLM.from_pretrained(
            model_path,
            dtype=torch.float16,
            device_map="cuda:0",
        )
        pbar.update(1)

        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            model_max_length=768,
        )
        pbar.update(1)

    # Load test dataset
    print("Loading dataset...")
    dataset = load_dataset('openai/gsm8k', 'main', split='test')
    if num_samples:
        dataset = dataset.select(range(num_samples))
    total_samples = len(dataset)
    print(f"Loaded {total_samples} samples")

    results = []
    correct = 0
    total = 0

    # Create progress bar
    progress_bar = tqdm(
        total=total_samples,
        desc="Processing samples",
        unit="examples",
        dynamic_ncols=True,
    )

    progress_bar.set_postfix({
        'acc': '0.00%',
        'correct': '0',
    })

    # Process in batches
    for i in range(0, total_samples, batch_size):
        batch_data = dataset[i:i + batch_size]
        current_batch_size = len(batch_data['question'])

        # Prepare prompts using same format as training
        prompts = [
            [
                {'role': 'system', 'content': R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS},
                {'role': 'user', 'content': "What is 2+2?"},
                {'role': 'assistant', 'content': "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>"},
                {'role': 'user', 'content': q.strip()}
            ] for q in batch_data['question']
        ]

        # Convert to chat format
        formatted_prompts = [
            tokenizer.apply_chat_template(
                p,
                return_tensors='pt',
                return_dict=True,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            for p in prompts
        ]

        # Generate responses (greedy decoding), keeping only the generated tokens
        responses = []
        for inputs in formatted_prompts:
            inputs = inputs.to('cuda:0')
            with torch.inference_mode():
                output = llm.generate(**inputs, max_new_tokens=512, do_sample=False)
            responses.append(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))

        # Process responses
        for j, response in enumerate(responses):

            # Extract answers
            generated_answer = extract_xml_answer(response)
            true_answer = extract_hash_answer(batch_data['answer'][j])

            # Store result
            result = {
                'question': batch_data['question'][j],
                'true_answer': true_answer,
                'generated_answer': generated_answer,
                'full_response': response,
                'correct': generated_answer == true_answer
            }
            results.append(result)

            # Update metrics
            if generated_answer == true_answer:
                correct += 1
            total += 1

        # Update progress
        progress_bar.update(current_batch_size)
        progress_bar.set_postfix({
            'acc': f'{(correct/total)*100:.2f}%',
            'correct': f'{correct}/{total}',
        })

    progress_bar.close()

    # Calculate metrics
    accuracy = correct / total if total > 0 else 0
    metrics = {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'model_path': model_path,
        'timestamp': datetime.now().isoformat()
    }

    # Save results
    if save_results:
        save_path = f"gsm8k_eval_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(save_path, 'w') as f:
            json.dump({
                'metrics': metrics,
                'results': results
            }, f, indent=2)
        print(f"\nResults saved to {save_path}")

    return metrics

print("Starting GSM8K evaluation...")

Starting GSM8K evaluation...


### Evaluation before GRPO

In [ ]:
before_metrics = evaluate_model(
    model_path="Qwen/Qwen3-0.6B",
    batch_size=1,
    num_samples=200,
)

Initializing evaluation...


Loading dataset...
Loaded 200 samples



Results saved to gsm8k_eval_results_20261018_205307.json


Processing samples: 200/200 [04:07<00:00, 1.16examples/s, acc=10.00%, correct=20/200]


### Evaluation after GRPO

In [ ]:
after_metrics = evaluate_model(
    model_path="grpo_model/final_merged_grpo_model",
    batch_size=1,
    num_samples=200,
)

Initializing evaluation...


Loading dataset...
Loaded 200 samples



Results saved to gsm8k_eval_results_20261018_211129.json


Processing samples: 200/200 [18:17<00:00, 4.67s/examples, acc=49.50%, correct=99/200]


In [ ]:
# Log comparison to W&B
wandb.init(
    project="ALTeGraD-lab3",
    name="GRPO-GSM8K-Evaluation",
    job_type="evaluation",
)

> **Question 1.** - What are the main conceptual and practical differences between **Direct Preference Optimization (DPO)** and **Group Relative Policy Optimization (GRPO)**?  
> - In what situations might one method outperform the other?  
> - Both methods relate to the **KL-regularized reward maximization objective** $\max_\pi \mathbb{E}_{x, y \sim \pi}[r(x, y)] - \beta\, \mathrm{KL}(\pi \,\|\, \pi_{\mathrm{ref}})$. How does each of them optimize it (or a variant of it)? Note that we set `beta=0` in our GRPO configuration: what does this change?
> 
> *References:*  
> - Rafailov et al., 2023. *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.* [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)

**Answer 1.**

### 1. Conceptual and practical differences

**Direct Preference Optimization (DPO)** is an offline preference-learning algorithm. Each training sample consists of a triplet $(x,y_w,y_l)$, where $x$ is the prompt, $y_w$ is the preferred response, and $y_l$ is the rejected response. DPO optimizes the policy to increase the relative likelihood of $y_w$ compared with $y_l$, without explicitly training a reward model or generating new responses during training.

**Group Relative Policy Optimization (GRPO)** is an online reinforcement learning algorithm. Given a prompt $x$, the current policy generates a group of $G$ responses $\{y_1,\dots,y_G\}$. Each response receives a reward $r_i=r(x,y_i)$, which is used to compute a group-relative advantage:

$$
A_i=\frac{r_i-\mu_r}{\sigma_r}
$$

GRPO updates the policy using these advantages through a PPO-style objective, without requiring a separate critic model.

In these experiments, DPO uses French Orca preference pairs, while GRPO uses GSM8K with rewards based on mathematical correctness and output format.

### 2. When is each method preferable?

- **DPO** is preferable when high-quality preference pairs are available, particularly for subjective tasks such as helpfulness or conversational quality. It is generally simpler and computationally cheaper.
- **GRPO** is preferable when rewards can be computed automatically, as in mathematical reasoning or code generation. It allows online exploration but requires more computation and a reliable reward function.

### 3. Connection to KL-regularized reward maximization

Both methods are related to the objective:

$$
\max_\pi\;
\mathbb{E}_{x,y\sim\pi}[r(x,y)]
-\beta D_{\mathrm{KL}}(\pi\|\pi_{\mathrm{ref}})
$$

The first term maximizes expected reward, while the KL penalty controls deviation from the reference policy.

**DPO** exploits the analytical relationship between the optimal policy and the reward:

$$
r(x,y)=\beta\log\frac{\pi^*(y|x)}{\pi_{\mathrm{ref}}(y|x)}+\beta\log Z(x)
$$

This leads to the DPO loss:

$$
\mathcal{L}_{\mathrm{DPO}}
=
-\mathbb{E}_{(x,y_w,y_l)}
\left[
\log\sigma\left(
\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
-
\beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\right)
\right]
$$

Thus, DPO optimizes preferences indirectly through a classification loss derived from the KL-regularized objective.

**GRPO** directly updates the policy using group-relative advantages and a clipped surrogate objective:

$$
\mathcal{J}_{\mathrm{GRPO}}
=
\mathbb{E}\left[
\min\left(
\rho A,\,
\operatorname{clip}(\rho,1-\epsilon,1+\epsilon)A
\right)
-\beta D_{\mathrm{KL}}
\right]
$$

where $\rho=\pi_\theta/\pi_{\mathrm{old}}$ is the token-level policy probability ratio. The expression is a simplified form of the objective, which averages over generated responses and tokens.

### 4. Effect of beta = 0 in GRPO

In our experiment, setting $\beta=0$ removes the explicit KL penalty:

$$
\mathcal{J}_{\mathrm{GRPO}}
=
\mathbb{E}\left[
\min\left(
\rho A,\,
\operatorname{clip}(\rho,1-\epsilon,1+\epsilon)A
\right)
\right]
$$

The model can therefore deviate more freely from the reference policy, and reference-model computations are no longer required for KL regularization. This reduces computational cost but increases the risk of policy drift.

However, **PPO-style clipping** ($\epsilon=0.28$ in our experiment) still constrains individual policy updates relative to the rollout policy.

> **Question 2.** - Why can **reward model overfitting** or **reward hacking** occur in reinforcement fine-tuning?  
> - How do DPO and our GRPO setup avoid relying on a learned reward model, and what risks remain?  
> - Discuss the role of the **reference model** in maintaining stability.

**Answer 2.**

### 1. Reward overfitting and reward hacking

In reinforcement fine-tuning, a learned reward model $r_\phi(x,y)$ approximates human preferences. Since this approximation is imperfect, the policy may exploit its weaknesses.

**Reward overfitting** occurs when the reward model learns spurious patterns, while **reward hacking** occurs when the policy maximizes the estimated reward without improving actual response quality:

$$
\arg\max_\pi \mathbb{E}[r_\phi]
\neq
\arg\max_\pi \mathbb{E}[r_{\mathrm{true}}]
$$

### 2. How DPO and GRPO avoid learned reward models

**DPO** directly optimizes preference pairs $(x,y_w,y_l)$ using a classification-based loss, without explicitly training a reward model. However, it remains sensitive to noisy or biased preference labels.

**GRPO**, in our GSM8K experiment, uses rule-based rewards instead of a learned reward model:

$$
r(x,y)=r_{\mathrm{format}}(y)+r_{\mathrm{correct}}(x,y)
$$

The model receives up to 3 points for correct formatting and numerical accuracy. However, it may still exploit weaknesses in the reward function or produce correct answers without reliable reasoning.

### 3. Role of the reference model

The reference policy $\pi_{\mathrm{ref}}$ stabilizes training by penalizing excessive deviations from the original model:

$$
\mathcal{J}(\pi)=\mathbb{E}[r(x,y)]-\beta D_{\mathrm{KL}}(\pi\|\pi_{\mathrm{ref}})
$$

In **DPO**, the reference policy is incorporated into the preference loss. In **GRPO**, the KL penalty is optional.

In our experiment, $\beta=0$ removes explicit KL regularization, reducing computational cost but increasing the risk of policy drift. PPO-style clipping ($\epsilon=0.28$) still limits individual updates, but does not guarantee proximity to the original policy.

> **Question 3.** - In the GSM8K setup, we used a simple binary reward for **format** and **correctness**.  
>   - What are the limitations of such sparse rewards?  
>   - How could you design a **richer, smoother** reward signal for math reasoning?

**Answer 3.**

### 1. Limitations of sparse rewards

In our GSM8K setup, the reward is defined as:

$$
r(x,y)=r_{\mathrm{format}}(y)+r_{\mathrm{correct}}(x,y)
$$

with $r_{\mathrm{format}}\in\{0,1\}$ and $r_{\mathrm{correct}}\in\{0,2\}$.

These **sparse rewards** provide limited feedback: a nearly correct solution and a completely incorrect solution can receive the same reward. They do not evaluate intermediate reasoning steps or partial progress.

Moreover, in GRPO, if all generated responses receive the same reward, their group-relative advantages are zero, providing no useful learning signal.

### 2. Designing a richer reward signal

A smoother reward could combine several components:

$$
r(x,y)=
\lambda_1 r_{\mathrm{format}}
+\lambda_2 r_{\mathrm{answer}}
+\lambda_3 r_{\mathrm{steps}}
+\lambda_4 r_{\mathrm{consistency}}
$$

where:

- $r_{\mathrm{format}}$: Gradual score for respecting the required output structure.
- $r_{\mathrm{answer}}$: Partial credit based on numerical proximity to the correct answer, for example:

$$
r_{\mathrm{answer}}=
\exp\left(-\frac{|\hat{a}-a|}{\tau\max(1,|a|)}\right)
$$

- $r_{\mathrm{steps}}$: Rewards mathematically valid intermediate reasoning steps.
- $r_{\mathrm{consistency}}$: Rewards consistency between calculations and the final answer.

Here, $\lambda_i$ control the importance of each component, and $\tau>0$ controls reward smoothness.

Such a reward provides more informative feedback and can improve learning efficiency. However, intermediate-step validation must be reliable, and numerical proximity alone does not guarantee correct mathematical reasoning.